<a href="https://colab.research.google.com/github/imtisalrangrez/DeepLearning_Lab/blob/main/DL_week8(180).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ZF-Net

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Load and Preprocess CIFAR-10
# ---------------------------------------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['Airplane','Automobile','Bird','Cat','Deer',
               'Dog','Frog','Horse','Ship','Truck']

# Already (batch, 32, 32, 3) — normalize only, no reshape needed
x_train = x_train / 255.0
x_test  = x_test  / 255.0

# ---------------------------------------------------------
# 2. Build ZF-Net for CIFAR-10
# ---------------------------------------------------------
inputs = layers.Input(shape=(32, 32, 3))

# Layer 1: 5x5 replaces 7x7 — gentler entry on 32x32 images
# strides=(2,2) mimics ZF-Net's original aggressive downsampling intent
x = layers.Conv2D(96, (5, 5), strides=(2, 2), padding='same', activation='relu')(inputs)
x = layers.MaxPooling2D((2, 2))(x)                     # → 8x8

# Layer 2: 5x5 filter — richer mid-level feature extraction for RGB
x = layers.Conv2D(256, (5, 5), padding='same', activation='relu')(x)
x = layers.MaxPooling2D((2, 2))(x)                     # → 4x4

# Layers 3-5: Stacked 3x3 convolutions — no pooling to preserve 4x4 spatial info
x = layers.Conv2D(384, (3, 3), padding='same', activation='relu')(x)
x = layers.Conv2D(384, (3, 3), padding='same', activation='relu')(x)
x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(x)

# ---------------------------------------------------------
# Classification Head — full 4096 capacity for CIFAR-10
# ---------------------------------------------------------
x = layers.Flatten()(x)
x = layers.Dense(4096, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(4096, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation='softmax')(x)    # 10 CIFAR-10 classes

model = Model(inputs, outputs, name="ZF-Net-CIFAR10")

# ---------------------------------------------------------
# 3. Compile
# ---------------------------------------------------------
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# patience=4 — CIFAR-10 val_loss plateaus more before truly diverging
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

# ---------------------------------------------------------
# 4. Train
# ---------------------------------------------------------
print("Training ZF-Net on CIFAR-10...")
history = model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    verbose=0,
    callbacks=[early_stop]
)

# ---------------------------------------------------------
# 5. Evaluate
# ---------------------------------------------------------
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test Loss     : {test_loss:.4f}')
model.summary()

# CIFAR-10 Architecture Progression:
#   LeNet-5  → ~65-68%
#   AlexNet  → ~78-82%
#   VGG      → ~83-87%
#   ZF-Net   → ~82-86%  (comparable to VGG, slightly lower due to fewer conv blocks)

# ---------------------------------------------------------
# 6. Plot Learning Curves
# ---------------------------------------------------------
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('ZF-Net Accuracy — CIFAR-10')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('ZF-Net Loss — CIFAR-10')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

#GoogLeNet (Inception V1)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Load and Preprocess CIFAR-10
# ---------------------------------------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['Airplane','Automobile','Bird','Cat','Deer',
               'Dog','Frog','Horse','Ship','Truck']

# Already (batch, 32, 32, 3) — normalize only
x_train = x_train / 255.0
x_test  = x_test  / 255.0

# ---------------------------------------------------------
# 2. Inception Block (unchanged — architecture is dataset-agnostic)
# ---------------------------------------------------------
def inception_block(x, f1, f3_in, f3, f5_in, f5, proj):
    # Branch 1: 1x1 Conv
    conv1 = layers.Conv2D(f1, (1, 1), padding='same', activation='relu')(x)

    # Branch 2: 1x1 bottleneck → 3x3 Conv
    conv3 = layers.Conv2D(f3, (3, 3), padding='same', activation='relu')(
            layers.Conv2D(f3_in, (1, 1), padding='same', activation='relu')(x))

    # Branch 3: 1x1 bottleneck → 5x5 Conv
    conv5 = layers.Conv2D(f5, (5, 5), padding='same', activation='relu')(
            layers.Conv2D(f5_in, (1, 1), padding='same', activation='relu')(x))

    # Branch 4: MaxPool → 1x1 projection
    pool_proj = layers.Conv2D(proj, (1, 1), padding='same', activation='relu')(
                layers.MaxPooling2D((3, 3), strides=1, padding='same')(x))

    return layers.Concatenate(axis=-1)([conv1, conv3, conv5, pool_proj])

# ---------------------------------------------------------
# 3. Build GoogLeNet for CIFAR-10
# ---------------------------------------------------------
inputs = layers.Input(shape=(32, 32, 3))

# Stem — 5x5 replaces 7x7, strides=(2,2) handles spatial reduction gently
x = layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', activation='relu')(inputs)
x = layers.BatchNormalization()(x)                     # stabilizes color gradient learning
x = layers.MaxPooling2D((2, 2))(x)                     # → 8x8

x = layers.Conv2D(192, (3, 3), padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)                     # → 4x4

# ---------------------------------------------------------
# 9 Inception Blocks (same filter config as Fashion MNIST — justified by CIFAR complexity)
# ---------------------------------------------------------
x = inception_block(x,  64,  96, 128, 16,  32,  32)   # Block 1
x = inception_block(x, 128, 128, 192, 32,  96,  64)   # Block 2
x = inception_block(x, 192,  96, 208, 16,  48,  64)   # Block 3
x = inception_block(x, 160, 112, 224, 24,  64,  64)   # Block 4
x = inception_block(x, 128, 128, 256, 24,  64,  64)   # Block 5
x = inception_block(x, 112, 144, 288, 32,  64,  64)   # Block 6
x = inception_block(x, 256, 160, 320, 32, 128, 128)   # Block 7
x = inception_block(x, 256, 160, 320, 32, 128, 128)   # Block 8
x = inception_block(x, 384, 192, 384, 48, 128, 128)   # Block 9

# Global Average Pooling replaces Dense layers — keeps parameter count low
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(10, activation='softmax')(x)    # 10 CIFAR-10 classes

model = Model(inputs, outputs, name="GoogLeNet-CIFAR10")

# ---------------------------------------------------------
# 4. Compile
# ---------------------------------------------------------
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

# ---------------------------------------------------------
# 5. Train
# ---------------------------------------------------------
print("Training GoogLeNet on CIFAR-10...")
history = model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    verbose=0,
    callbacks=[early_stop]
)

# ---------------------------------------------------------
# 6. Evaluate
# ---------------------------------------------------------
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test Loss     : {test_loss:.4f}')
model.summary()

# CIFAR-10 Architecture Progression:
#   LeNet-5    → ~65-68%
#   AlexNet    → ~78-82%
#   ZF-Net     → ~82-86%
#   VGG        → ~83-87%
#   GoogLeNet  → ~87-91%  ← Inception blocks + GAP = big efficiency gain

# ---------------------------------------------------------
# 7. Plot Learning Curves
# ---------------------------------------------------------
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('GoogLeNet Accuracy — CIFAR-10')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('GoogLeNet Loss — CIFAR-10')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

#ResNet

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Load and Preprocess CIFAR-10
# ---------------------------------------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['Airplane','Automobile','Bird','Cat','Deer',
               'Dog','Frog','Horse','Ship','Truck']

# Already (batch, 32, 32, 3) — normalize only
x_train = x_train / 255.0
x_test  = x_test  / 255.0

# ---------------------------------------------------------
# 2. ResNet Block (with BatchNorm — standard for CIFAR ResNet)
# ---------------------------------------------------------
def resnet_block(x, filters, downsample=False):
    shortcut = x
    stride = 2 if downsample else 1

    x = layers.Conv2D(filters, (3, 3), strides=stride, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters, (3, 3), strides=1, padding='same')(x)
    x = layers.BatchNormalization()(x)

    # Projection shortcut when downsampling to match dimensions
    if downsample:
        shortcut = layers.Conv2D(filters, (1, 1), strides=2, padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    return layers.Activation('relu')(x)

# ---------------------------------------------------------
# 3. Build ResNet-18 for CIFAR-10
# ---------------------------------------------------------
inputs = layers.Input(shape=(32, 32, 3))

# FIX 1: Data Augmentation — model sees slightly different images every epoch
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomTranslation(height_factor=0.1, width_factor=0.1)(x)
x = layers.RandomZoom(height_factor=0.1)(x)

# Stem — 3x3 replaces 7x7, NO MaxPooling (32x32 can't afford early spatial loss)
x = layers.Conv2D(64, (3, 3), padding='same')(x)       # → 32x32
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)

# Stage 1 — 64 filters, no downsampling
x = resnet_block(x, 64)
x = resnet_block(x, 64)                                # → 32x32

# Stage 2 — 128 filters, downsample
x = resnet_block(x, 128, downsample=True)
x = resnet_block(x, 128)                               # → 16x16

# Stage 3 — 256 filters, downsample
x = resnet_block(x, 256, downsample=True)
x = resnet_block(x, 256)                               # → 8x8

# Stage 4 — 512 filters, downsample
x = resnet_block(x, 512, downsample=True)
x = resnet_block(x, 512)                               # → 4x4

# Global Average Pooling — kills memorization by averaging 512 feature maps
x = layers.GlobalAveragePooling2D()(x)

# FIX 2: Dropout before output — forces robust multi-feature learning
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation='softmax')(x)    # 10 CIFAR-10 classes

model = Model(inputs, outputs, name="ResNet-18-CIFAR10")

# ---------------------------------------------------------
# 4. Compile
# ---------------------------------------------------------
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# FIX 3: patience=5 — ResNet loss curves are bumpy, give it more room
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# ---------------------------------------------------------
# 5. Train
# ---------------------------------------------------------
print("Training ResNet-18 on CIFAR-10...")
history = model.fit(
    x_train, y_train,
    epochs=20,                  # more epochs — augmentation slows convergence slightly
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop]
)

# ---------------------------------------------------------
# 6. Evaluate
# ---------------------------------------------------------
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test Loss     : {test_loss:.4f}')
model.summary()

# CIFAR-10 Architecture Progression:
#   LeNet-5    → ~65-68%
#   AlexNet    → ~78-82%
#   ZF-Net     → ~82-86%
#   VGG        → ~83-87%
#   GoogLeNet  → ~87-91%
#   ResNet-18  → ~91-94%  ← residual shortcuts + BN + augmentation

# ---------------------------------------------------------
# 7. Plot Learning Curves
# ---------------------------------------------------------
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('ResNet-18 Accuracy — CIFAR-10')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('ResNet-18 Loss — CIFAR-10')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()